## This notebook is created by Prateek Paul.
* Email: prateekp@iiitd.ac.in
* LinkedIn: [linkedin.com/in/prateekpaulpro/](https://linkedin.com/in/prateekpaulpro/)

Disclaimer: 
The code and content in this notebook are intended solely for educational purposes. All credits for original ideas go to their respective authors.

# Bio Computing Course - Tutorial 2

# Topic: Sequence Alignment and Scoring Basics
### Welcome to Tutorial 2! In this tutorial, we will explore fundamental concepts in sequence alignment, including scoring matches, mismatches, and gaps. We will also implement simple dynamic programming matrices.

### 1. Simple Scoring Function
#### Question: Write a Python function `score_alignment(seq1, seq2, match_score, mismatch_penalty, gap_penalty)` that takes two aligned sequences of the same length (with gaps represented by '-') and calculates the total alignment score.

In [ ]:
def score_alignment(seq1, seq2, match, mismatch, gap):
    score = 0
    for a, b in zip(seq1, seq2):
        if a == '-' or b == '-':
            score += gap
        elif a == b:
            score += match
        else:
            score += mismatch
    return score

seq1 = "ATGC-TAC"
seq2 = "AT-CGTAC"
print("Score:", score_alignment(seq1, seq2, 2, -1, -2))

Score: 3


**Explanation**: We iterate through both aligned strings simultaneously using `zip()`. If either character is a gap `-`, we add the gap penalty. If they match, we add the match score. Otherwise, they mismatch, so we add the mismatch penalty.

### 2. Hamming Distance
#### Question: The Hamming distance between two strings of equal length is the number of positions at which the corresponding symbols are different. Write a function `hamming_distance(seq1, seq2)` that calculates this distance for two DNA sequences.

In [ ]:
def hamming_distance(seq1, seq2):
    if len(seq1) != len(seq2):
        raise ValueError("Sequences must be of equal length.")
    return sum(1 for a, b in zip(seq1, seq2) if a != b)

print("Hamming Distance:", hamming_distance("GATTACA", "GACTATA"))

Hamming Distance: 2


**Explanation**: The Hamming distance counts the exact number of character mismatches between two equal-length strings. Using a generator expression with `zip()` makes this extremely concise and Pythonic.

### 3. Identity Matrix Initialization
#### Question: In dynamic programming algorithms like Needleman-Wunsch, the first step is to initialize a scoring matrix. Write a function `initialize_matrix(len1, len2, gap_penalty)` that creates a 2D list (matrix) of size `(len1+1) x (len2+1)` and fills the first row and first column with gap penalties.

In [ ]:
def initialize_matrix(len1, len2, gap_penalty):
    # Create a zero-filled matrix of size (len1+1) x (len2+1)
    matrix = [[0 for _ in range(len2 + 1)] for _ in range(len1 + 1)]
    
    # Fill first column
    for i in range(len1 + 1):
        matrix[i][0] = i * gap_penalty
        
    # Fill first row
    for j in range(len2 + 1):
        matrix[0][j] = j * gap_penalty
        
    return matrix

matrix = initialize_matrix(3, 4, -2)
for row in matrix:
    print(row)

[0, -2, -4, -6, -8]
[-2, 0, 0, 0, 0]
[-4, 0, 0, 0, 0]
[-6, 0, 0, 0, 0]


**Explanation**: In global alignment (Needleman-Wunsch), the top row and left column represent aligning one sequence entirely against gaps. Therefore, the score at index `i` is simply `i * gap_penalty`.

### 4. Sequence Identity Percentage
#### Question: Write a function `calculate_identity(seq1, seq2)` that takes two aligned sequences of equal length and returns the percentage of identical characters (excluding gaps). If both characters are gaps '-', do not count them in the total length.

In [ ]:
def calculate_identity(seq1, seq2):
    matches = 0
    valid_length = 0
    for a, b in zip(seq1, seq2):
        if a == '-' and b == '-':
            continue
        valid_length += 1
        if a == b:
            matches += 1
    if valid_length == 0:
        return 0.0
    return (matches / valid_length) * 100

print(f"Identity: {calculate_identity('ATGC-T', 'AT-CGT'):.2f}%")

Identity: 50.00%


**Explanation**: We iterate through the aligned pairs. We ignore columns where both are gaps. We count the matches and divide by the total number of valid aligned positions to get the percentage.

### 5. Transition vs Transversion Counter
#### Question: In DNA, transitions (A<->G, C<->T) are more common than transversions (purine <-> pyrimidine). Write a function `count_mutations(seq1, seq2)` that compares two un-gapped, equal-length sequences and returns a dictionary with the counts of 'transitions' and 'transversions'.

In [ ]:
def count_mutations(seq1, seq2):
    purines = {'A', 'G'}
    pyrimidines = {'C', 'T'}
    counts = {'transitions': 0, 'transversions': 0}
    
    for a, b in zip(seq1, seq2):
        if a != b:
            if (a in purines and b in purines) or (a in pyrimidines and b in pyrimidines):
                counts['transitions'] += 1
            else:
                counts['transversions'] += 1
    return counts

print(count_mutations('ATGC', 'ACGT'))

{'transitions': 1, 'transversions': 3}


**Explanation**: We define sets for purines and pyrimidines. If a mismatch occurs within the same group, it's a transition. If it crosses groups, it's a transversion.

### 6. Needleman-Wunsch Matrix Scoring
#### Question: Write a function `fill_nw_matrix(seq1, seq2, match, mismatch, gap)` that fully populates the Needleman-Wunsch dynamic programming matrix. You can use your `initialize_matrix` logic from Question 3 as a starting point. Return the completed matrix.

In [ ]:
def fill_nw_matrix(seq1, seq2, match, mismatch, gap):
    rows, cols = len(seq1) + 1, len(seq2) + 1
    matrix = [[0 for _ in range(cols)] for _ in range(rows)]
    
    for i in range(rows): matrix[i][0] = i * gap
    for j in range(cols): matrix[0][j] = j * gap
    
    for i in range(1, rows):
        for j in range(1, cols):
            score_diag = matrix[i-1][j-1] + (match if seq1[i-1] == seq2[j-1] else mismatch)
            score_up = matrix[i-1][j] + gap
            score_left = matrix[i][j-1] + gap
            matrix[i][j] = max(score_diag, score_up, score_left)
            
    return matrix

mat = fill_nw_matrix('AT', 'AG', 1, -1, -1)
for r in mat: print(r)

[0, -1, -2]
[-1, 1, 0]
[-2, 0, 0]


**Explanation**: We iterate through each cell `(i, j)`. The score is the maximum of three possible moves: coming from a diagonal (match/mismatch), coming from above (gap in seq2), or coming from the left (gap in seq1).

### 7. Smith-Waterman Local Alignment (Finding the Max)
#### Question: Unlike global alignment, Smith-Waterman (local alignment) resets negative scores to 0. Write a function `find_sw_max(matrix)` that takes a completed Smith-Waterman scoring matrix (2D list) and returns a tuple `(max_score, (row_index, col_index))` indicating the start of the traceback.

In [ ]:
def find_sw_max(matrix):
    max_val = 0
    max_pos = (0, 0)
    
    for i, row in enumerate(matrix):
        for j, val in enumerate(row):
            if val > max_val:
                max_val = val
                max_pos = (i, j)
                
    return max_val, max_pos

# Mock matrix for testing
mock_matrix = [[0, 0, 0], [0, 2, 0], [0, 0, 4]]
print("Max score and position:", find_sw_max(mock_matrix))

Max score and position: (4, (2, 2))


**Explanation**: In local alignment, the optimal alignment ends at the highest score in the entire matrix. We simply iterate through the 2D array to find the maximum value and its coordinates.

### 8. Generating an Alignment String
#### Question: Often, alignments are visualized with a middle string showing pipes `|` for matches, spaces for mismatches, and gaps. Write `visualize_alignment(seq1, seq2)` that prints this 3-line visualization for two aligned sequences.

In [ ]:
def visualize_alignment(seq1, seq2):
    middle = []
    for a, b in zip(seq1, seq2):
        if a == b and a != '-':
            middle.append('|')
        else:
            middle.append(' ')
            
    print(seq1)
    print(''.join(middle))
    print(seq2)

visualize_alignment("ATGC-TAC", "AT-CGTAC")

ATGC-TAC
||   |||
AT-CGTAC


**Explanation**: We compare the two strings character by character. If they match (and aren't gaps), we place a pipe. Otherwise, we leave a space. This makes it easy for humans to spot conserved regions.